# 02 — Preprocessing
Clean the data, keep ALL 15 features and ALL 114 genres, encode labels, scale, and split.
We'll use feature importances from training to decide what to cut in a future iteration.

In [ ]:
# Cell 1 — Imports
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Cell 2 — Load full dataset (no genre filtering this time)
df = pd.read_csv("../data/SpotifySongs.csv", index_col=0)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Genres: {df['track_genre'].nunique()} unique")

In [ ]:
# Cell 3 — Clean: drop duplicates and null rows
# Duplicates can bias the model; nulls will crash scikit-learn
before = len(df)

df = df.drop_duplicates()
dupes_removed = before - len(df)

df = df.dropna()
nulls_removed = before - dupes_removed - len(df)

print(f"Duplicates removed : {dupes_removed:,}")
print(f"Null rows removed  : {nulls_removed:,}")
print(f"Rows remaining     : {len(df):,}")

In [ ]:
# Cell 4 — Feature selection: keep ALL numeric features + target
# We intentionally include everything here. Feature importances from the
# Random Forest (notebook 03) will tell us what's actually useful.
ALL_FEATURES = [
    'popularity',
    'duration_ms',
    'mode',
    'danceability',
    'energy',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'loudness',
    'speechiness',
    'acousticness',
]
TARGET = 'track_genre'

df = df[ALL_FEATURES + [TARGET]]
print(f"Columns kept: {df.columns.tolist()}")
df.head()

In [ ]:
GENRES_30 = [
    # Rock family
    'rock', 'alt-rock', 'alternative', 'punk-rock', 'black-metal', 'death-metal', 'psych-rock', 'hard-rock',
    
    # Electronic family  
    'electronic', 'techno', 'trance', 'club', 'chicago-house', 'deep-house', 'idm', 'breakbeat',
    
    # Urban/rhythmic
    'hip-hop', 'rap', 'r-n-b', 'dancehall',
    
    # Acoustic/organic
    'acoustic', 'folk', 'classical', 'jazz', 'blues',
    
    # Pop/mainstream
    'pop', 'dance', 'indie', 'soul', 'funk',
]

df = df[df['track_genre'].isin(GENRES_30)]
print(f"Rows remaining: {len(df)}")
print(f"Genres remaining: {df['track_genre'].nunique()}")

In [ ]:
df['track_genre'] = df['track_genre'].replace({
    'alt-rock': 'alternative',        # bleeding into each other
    'psych-rock': 'alternative',      # bleeding into alt-rock/alternative
    'hard-rock': 'rock',              # bleeding into rock
    'punk-rock': 'rock',              # bleeding into rock
    'death-metal': 'black-metal',     # bleeding into each other
    'breakbeat': 'electronic',        # bleeding into electronic
    'deep-house': 'chicago-house',    # bleeding into each other
    'idm': 'electronic',              # bleeding into electronic
    'dub': 'electronic',              # weak diagonal
    'soul': 'r-n-b',                  # bleeding into r-n-b
})

df['track_genre'] = df['track_genre'].replace({
    'indie': 'alternative',      # bleeding into alternative
    'club': 'electronic',        # bleeding into electronic
    'dance': 'electronic',       # bleeding into electronic
    'funk': 'r-n-b',             # bleeding into r-n-b
    'dub': 'electronic',         # weak diagonal, scattered
})

df['track_genre'] = df['track_genre'].replace({
    'folk': 'acoustic',       # both low energy, organic
    'blues': 'jazz',          # closer sonically than blues→rock
    'r-n-b': 'hip-hop',       # bleeding heavily into each other
    'rock': 'alternative',    # bleeding into alternative
})

print(df['track_genre'].nunique())
print(df['track_genre'].value_counts())

In [ ]:
SAMPLES_PER_GENRE = 2000

sampled_dfs = []
for genre in df['track_genre'].unique():
    genre_df = df[df['track_genre'] == genre]
    sampled_dfs.append(genre_df.sample(min(len(genre_df), SAMPLES_PER_GENRE), random_state=42))

df = pd.concat(sampled_dfs).reset_index(drop=True)
print(df['track_genre'].value_counts())

In [ ]:
print(df.columns.tolist())
print(df['track_genre'].unique())

In [ ]:
from sklearn.metrics import silhouette_samples
from sklearn.preprocessing import LabelEncoder
import pandas as pd

le = LabelEncoder()
y_encoded = le.fit_transform(df['track_genre'])

sil_scores = silhouette_samples(df[ALL_FEATURES], y_encoded)

df['silhouette'] = sil_scores
genre_sil = df.groupby('track_genre')['silhouette'].mean().sort_values()

print(genre_sil.head(30))  # worst genres

In [ ]:
# Cell 5 — Encode target labels
# LabelEncoder turns genre strings into integers (e.g. 'acoustic' → 0, 'afrobeat' → 1, ...)
# We save the encoder so we can convert predictions back to genre names later
le = LabelEncoder()
y = le.fit_transform(df[TARGET])

print(f"Total classes: {len(le.classes_)}")
print("\nLabel encoding map (first 20):")
for i, genre in enumerate(le.classes_[:20]):
    print(f"  {i:>3} → {genre}")
print("  ...")

In [ ]:
# Cell 6 — Train/test split, then scale features
# IMPORTANT: scaler is fit ONLY on training data to prevent data leakage.
# Leakage = the model accidentally "sees" test data during training → inflated scores.
X = df[ALL_FEATURES].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keeps genre proportions equal in both splits
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit + transform on train
X_test  = scaler.transform(X_test)       # transform only on test

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

In [ ]:
# Cell 7 — Verify class distribution in each split
# Stratify should keep counts balanced; this confirms it worked
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()

dist = pd.DataFrame({
    'genre'      : le.classes_,
    'train_count': train_counts.values,
    'test_count' : test_counts.values
})
print(dist.to_string(index=False))

In [ ]:
# Cell 8 — Save processed arrays, scaler, and encoder
# Notebooks 03 and 04 load these files instead of re-running preprocessing
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/X_test.npy",  X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy",  y_test)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(le,     "../models/label_encoder.pkl")

print("Saved:")
print("  data/processed/X_train.npy")
print("  data/processed/X_test.npy")
print("  data/processed/y_train.npy")
print("  data/processed/y_test.npy")
print("  models/scaler.pkl")
print("  models/label_encoder.pkl")